In [1]:
"""
This book scraps mockdraftable for all players combine stats from 1999 - 2026
Output is a csv file name 'nfl_combine_df.csv'
"""

"\nThis book scraps mockdraftable for all players combine stats from 1999 - 2026\nOutput is a csv file name 'nfl_combine_df.csv'\n"

In [2]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import requests
import re
import time
from pathlib import Path
import os

In [3]:
# Constants

FRACTIONS = {
    "½": ".5", "¼": ".25", "¾": ".75",
    "⅛": ".125", "⅜": ".375", "⅝": ".625", "⅞": ".875",
    "⅓": ".333", "⅔": ".667",
}


PAGE_URL_START_YEAR = 1999
PAGE_URL_END_YEAR = 2026
PAGE_URL_BASE = 'https://www.mockdraftable.com/'
NUMBER_OF_PAGES =  437
LIST_OF_PAGE_URLS = [f'{PAGE_URL_BASE}search?position=ATH&beginYear={PAGE_URL_START_YEAR}&endYear={PAGE_URL_END_YEAR}&sort=DESC&page={page_num}' for page_num in range(1, NUMBER_OF_PAGES+1)]

In [4]:
# Paths

try:
    main_directory = Path(__file__).parent
except NameError:
    main_directory = Path(os.path.dirname(os.getcwd()))  # fallback for Jupyter

data_folder = main_directory / 'data' # data folder output
notebook_folder = main_directory / 'notebooks' 

In [5]:
def parse_college_draft(text):
    '''
    Helper function to extract college and draft year from a player card
    '''
    
    if not text:
        return None, None

    parts = [x.strip() for x in text.split(',')]

    if len(parts) == 2:
        return parts[0], parts[1]
    elif len(parts) == 1:
        return None, parts[0]
    else:
        return None, None

def replace_fractions(val: str) -> str:
    '''
    Helper fucntion to get rid of fractions in data
    '''
    for frac, dec in FRACTIONS.items():
        val = val.replace(frac, dec)
    return val


def extract_number(val: str) -> float | None:
    '''
    Strip units/symbols and return a float, or None if unparseable
    '''
    val = str(val).strip()
    if val in ("", "nan", "None", "N/A", "-"):
        return None
    val = replace_fractions(val)
    match = re.search(r"[\d]+\.?[\d]*", val)
    return float(match.group()) if match else None

def parse_height(val: str) -> float | None:
    '''
    Handle Height
    '''
    val = str(val).strip()
    if val in ("", "nan"):
        return None
    val = replace_fractions(val)
    m = re.match(r"(\d+)['\u2019](\d*\.?\d*)", val)   # e.g. 6'4 or 6'4.5
    if m:
        feet, inches = m.groups()
        return int(feet) * 12 + (float(inches) if inches else 0.0)
    m2 = re.search(r"[\d]+\.?[\d]*", val)              # bare number -> inches
    return float(m2.group()) if m2 else None

In [6]:
# Loop through each page and get player_name, draft year, draf position, college, player_id

with requests.Session() as session:
    player_info_dict = {}
    for page_index, page_url in enumerate(LIST_OF_PAGE_URLS):
        page_num = page_index+1
        
        try:
            page_text = session.get(page_url, timeout=10).text
            soup = BeautifulSoup(page_text, 'html.parser')
        except requests.RequestException as e:
            print(f'Failed to access page {page_num} | Error {e}')
            continue  
        
        print(f'{"-"*10} PAGE {page_num} {"-"*10}')
        
        if page_num % 50 == 0:
            pd.DataFrame(player_info_dict.values()).to_csv(data_folder / 'md_progress_backup.csv', index=False)

        # Get table of players
        players_table = soup.find('div', class_ = 'list-group mb-2')
        if not players_table:
            print(f"No table found on page {page_num}")
            continue

        # Get list of player cards from table
        players_cards = players_table.find_all(class_='list-group-item list-group-item-action justify-content-between d-flex')

        # Get Name + player link from player_cards
        for player in players_cards:
            
            span_split_college_draft_class = player.find('span', class_='align-middle ml-2').get_text(strip=True)
            # Base for college + draft
            college, draft_class = parse_college_draft(span_split_college_draft_class) if span_split_college_draft_class else ("",-1) # Get college and draft year

            h5_player_name = player.find('h5', class_='list-group-item-heading mb-0 text-dark')
            player_name = h5_player_name.get_text(strip=True) if h5_player_name else "" # Player's name
            clean_player_name = re.sub('[^A-Za-z0-9]+', '', player_name) if h5_player_name else "" # Strip out everything but letters
            player_id = clean_player_name.lower() + str(draft_class)  if h5_player_name else "" # Create an id clean_name + draft year

            href = player.get('href') # Link to Player page
            if href:
                link_to_players_page = PAGE_URL_BASE + href
            else:
                link_to_players_page = ""
            
            span_draft_position = player.find('span', class_ ='badge') # Draft position
            draft_position = span_draft_position.get('title') if span_draft_position else ""

            # print(f'player_info | {player_name}\n\t player_id {player_id} \n\t link_to_page {link_to_players_page} \n\t draft_position {draft_position} \n\t college {college} \n\t draft_class {draft_class}')

            print(f'\tAdded {player_name}')
            player_dict = {
                'player_name': player_name,
                'player_id' : player_id,
                'player_url': link_to_players_page,
                'draft_class': int(draft_class),
                'position': draft_position,
                'college': college,
            }

            if player_id in player_info_dict:
                continue
            else:
                player_info_dict[player_id] = player_dict
        
        time.sleep(1)
        print(f'\tTotal Players added {len(players_cards)}\n')
        print(f'')
        

    print(f'{"-"*10}\nNum of players added: {len(player_info_dict)}{"-"*10}')


    # else:


    



---------- PAGE 1 ----------
	Added A'Shawn Robinson
	Added A.C. Leonard
	Added A.J. Brown
	Added A.J. Cann
	Added A.J. Derby
	Added A.J. Edds
	Added A.J. Epenesa
	Added A.J. Green
	Added A.J. Green
	Added A.J. Greene
	Added A.J. Hawk
	Added A.J. Jefferson
	Added A.J. Jenkins
	Added A.J. Klein
	Added A.J. McCarron
	Added A.J. Nicholson
	Added A.J. Stamps
	Added A.J. Terrell
	Added A.Q. Shipley
	Added AJ Barner
	Total Players added 20


---------- PAGE 2 ----------
	Added AJ Haulcy
	Added AT Perry
	Added Aamil Wagner
	Added Aaron  Lynch
	Added Aaron Anderson
	Added Aaron Banks
	Added Aaron Brooks
	Added Aaron Burbridge
	Added Aaron Casey
	Added Aaron Colvin
	Added Aaron Corp
	Added Aaron Curry
	Added Aaron Dalan
	Added Aaron Davis
	Added Aaron Dobson
	Added Aaron Donald
	Added Aaron Fairooz
	Added Aaron Fields
	Added Aaron Francisco
	Added Aaron Fuller
	Total Players added 20


---------- PAGE 3 ----------
	Added Aaron Gibson
	Added Aaron Hansford
	Added Aaron Hester
	Added Aaron Humphr

In [7]:
display(player_info_dict)

{'ashawnrobinson2016': {'player_name': "A'Shawn Robinson",
  'player_id': 'ashawnrobinson2016',
  'player_url': 'https://www.mockdraftable.com//player/ashawn-robinson?position=ATH',
  'draft_class': 2016,
  'position': 'Defensive Tackle',
  'college': 'Alabama'},
 'acleonard2014': {'player_name': 'A.C. Leonard',
  'player_id': 'acleonard2014',
  'player_url': 'https://www.mockdraftable.com//player/ac-leonard?position=ATH',
  'draft_class': 2014,
  'position': 'Tight End',
  'college': 'Tennessee State'},
 'ajbrown2019': {'player_name': 'A.J. Brown',
  'player_id': 'ajbrown2019',
  'player_url': 'https://www.mockdraftable.com//player/aj-brown?position=ATH',
  'draft_class': 2019,
  'position': 'Wide Receiver',
  'college': 'Ole Miss'},
 'ajcann2015': {'player_name': 'A.J. Cann',
  'player_id': 'ajcann2015',
  'player_url': 'https://www.mockdraftable.com//player/aj-cann?position=ATH',
  'draft_class': 2015,
  'position': 'Offensive Guard',
  'college': 'South Carolina'},
 'ajderby2015': 

In [8]:
# Loop through each player page and extract height, wieght, arm length, hand size, 40 yard dash, vert jump, broad jump, bench press

with requests.Session() as session:
    for player_id in player_info_dict:
            player_url = player_info_dict[player_id]['player_url']
            player_name = player_info_dict[player_id]['player_name']
            try:
                player_page_text = session.get(player_url, timeout=10).text
            except requests.RequestException as e:
                 print(f'Failed to access player page for {player_name} | {e}')
                 continue
            
            time.sleep(0.5)
            player_soup = BeautifulSoup(player_page_text, 'html.parser')
            player_measurements_table  = player_soup.find('table', class_ = 'table table-sm mb-0')
            if not player_measurements_table:
                 print(f'No Measurements found for {player_name}')
                 continue
            measurements_list = player_measurements_table.find_all('tr')

            print(f'{"-"*10} {player_name.upper()}')
            for measurement in measurements_list[1:]:
                line_items = measurement.find_all('td')
                measurement_name = line_items[0].get_text(strip=True).lower()
                value = line_items[1].get_text(strip=True)

                print(f'{measurement_name} | {value}')
                player_info_dict[player_id][measurement_name] = value
                # print(measurement)
            # counter += 1

        # player_dict = {}


---------- A'SHAWN ROBINSON
height | 6' 4"
weight | 307 lbs
arm length | 34½"
hand size | 10½"
10 yard split | 1.78s
40 yard dash | 5.2s
vertical jump | 26"
broad jump | 106"
3-cone drill | 7.8s
20 yard shuttle | 4.74s
bench press | 22 reps
---------- A.C. LEONARD
height | 6' 2"
weight | 252 lbs
arm length | 33"
hand size | 9¼"
40 yard dash | 4.5s
vertical jump | 34"
broad jump | 128"
bench press | 20 reps
---------- A.J. BROWN
height | 6' 0½"
weight | 226 lbs
wingspan | 78"
arm length | 32⅞"
hand size | 9¾"
40 yard dash | 4.49s
vertical jump | 36½"
broad jump | 120"
bench press | 19 reps
---------- A.J. CANN
height | 6' 3"
weight | 313 lbs
arm length | 32⅝"
hand size | 10¼"
bench press | 26 reps
---------- A.J. DERBY
height | 6' 4"
weight | 255 lbs
arm length | 30½"
hand size | 9½"
bench press | 15 reps
---------- A.J. EDDS
height | 6' 4"
weight | 246 lbs
arm length | 32¾"
hand size | 9⅛"
10 yard split | 1.6s
40 yard dash | 4.62s
vertical jump | 33"
broad jump | 117"
3-cone drill | 7.

In [9]:
display(player_info_dict)

{'ashawnrobinson2016': {'player_name': "A'Shawn Robinson",
  'player_id': 'ashawnrobinson2016',
  'player_url': 'https://www.mockdraftable.com//player/ashawn-robinson?position=ATH',
  'draft_class': 2016,
  'position': 'Defensive Tackle',
  'college': 'Alabama',
  'height': '6\' 4"',
  'weight': '307 lbs',
  'arm length': '34½"',
  'hand size': '10½"',
  '10 yard split': '1.78s',
  '40 yard dash': '5.2s',
  'vertical jump': '26"',
  'broad jump': '106"',
  '3-cone drill': '7.8s',
  '20 yard shuttle': '4.74s',
  'bench press': '22 reps'},
 'acleonard2014': {'player_name': 'A.C. Leonard',
  'player_id': 'acleonard2014',
  'player_url': 'https://www.mockdraftable.com//player/ac-leonard?position=ATH',
  'draft_class': 2014,
  'position': 'Tight End',
  'college': 'Tennessee State',
  'height': '6\' 2"',
  'weight': '252 lbs',
  'arm length': '33"',
  'hand size': '9¼"',
  '40 yard dash': '4.5s',
  'vertical jump': '34"',
  'broad jump': '128"',
  'bench press': '20 reps'},
 'ajbrown2019': 

In [10]:
nfl_combine_df = pd.DataFrame(player_info_dict.values())

nfl_combine_df_rename_cols = {
    'player_name':'md_player_name',
    'player_id':'md_player_id',
    'player_url':'md_url',
    'draft_class':'md_draft_class',
    'position':'md_position',
    'college':'md_college',
    'height':'md_height',
    'weight':'md_weight',
    'arm length':'md_arm_length',
    'hand size':'md_hand_size',
    'wingspan':'md_wing_span',
    '10 yard split':'md_yard_10_split',
    '20 yard split': 'md_yard_20_split',
    '40 yard dash':'md_yard_40_dash',
    'vertical jump': 'md_vert_jump',
    'broad jump':'md_broad_jump',
    '3-cone drill':'md_cone_drill',
    '20 yard shuttle':'md_yard_20_shuttle',
    '60 yard shuttle':'md_yard_60_shuttle',
    'bench press':'md_bench_press'
}

nfl_combine_df = nfl_combine_df.rename(columns=nfl_combine_df_rename_cols)


In [11]:
display(nfl_combine_df.head())

for c in nfl_combine_df.columns:
    print(c)

,md_player_name,md_player_id,md_url,md_draft_class,md_position,md_college,md_height,md_weight,md_arm_length,md_hand_size,md_yard_10_split,md_yard_40_dash,md_vert_jump,md_broad_jump,md_cone_drill,md_yard_20_shuttle,md_bench_press,md_wing_span,md_yard_20_split,md_yard_60_shuttle
0,A'Shawn Robinson,ashawnrobinson2016,https://www.mockdraftable.com//player/ashawn-r...,2016,Defensive Tackle,Alabama,"6' 4""",307 lbs,"34½""","10½""",1.78s,5.2s,"26""","106""",7.8s,4.74s,22 reps,NaN,NaN,NaN
1,A.C. Leonard,acleonard2014,https://www.mockdraftable.com//player/ac-leona...,2014,Tight End,Tennessee State,"6' 2""",252 lbs,"33""","9¼""",NaN,4.5s,"34""","128""",NaN,NaN,20 reps,NaN,NaN,NaN
2,A.J. Brown,ajbrown2019,https://www.mockdraftable.com//player/aj-brown...,2019,Wide Receiver,Ole Miss,"6' 0½""",226 lbs,"32⅞""","9¾""",NaN,4.49s,"36½""","120""",NaN,NaN,19 reps,"78""",NaN,NaN
3,A.J. Cann,ajcann2015,https://www.mockdraftable.com//player/aj-cann?...,2015,Offensive Guard,South Carolina,"6' 3""",313 lbs,"32⅝""","10¼""",NaN,NaN,NaN,NaN,NaN,NaN,26 reps,NaN,NaN,NaN
4,A.J. Derby,ajderby2015,https://www.mockdraftable.com//player/aj-derby...,2015,Tight End,Arkansas,"6' 4""",255 lbs,"30½""","9½""",NaN,NaN,NaN,NaN,NaN,NaN,15 reps,NaN,NaN,NaN


md_player_name
md_player_id
md_url
md_draft_class
md_position
md_college
md_height
md_weight
md_arm_length
md_hand_size
md_yard_10_split
md_yard_40_dash
md_vert_jump
md_broad_jump
md_cone_drill
md_yard_20_shuttle
md_bench_press
md_wing_span
md_yard_20_split
md_yard_60_shuttle


In [12]:
nfl_combine_df.dtypes

md_player_name        object
md_player_id          object
md_url                object
md_draft_class         int64
md_position           object
md_college            object
md_height             object
md_weight             object
md_arm_length         object
md_hand_size          object
md_yard_10_split      object
md_yard_40_dash       object
md_vert_jump          object
md_broad_jump         object
md_cone_drill         object
md_yard_20_shuttle    object
md_bench_press        object
md_wing_span          object
md_yard_20_split      object
md_yard_60_shuttle    object
dtype: object

In [13]:


for col in nfl_combine_df.select_dtypes(include="object").columns:
    nfl_combine_df[col] = nfl_combine_df[col].astype(str).apply(replace_fractions)

In [14]:
nfl_combine_df["md_weight_lbs"] = nfl_combine_df["md_weight"].apply(extract_number)

In [15]:
nfl_combine_df["md_height_in"] = nfl_combine_df["md_height"].apply(parse_height)

In [16]:
for col in ["md_arm_length", "md_hand_size", "md_wing_span"]:
    nfl_combine_df[col + "_in"] = nfl_combine_df[col].apply(extract_number)

# 40-dash, 10-yd split, cone drill, 20-yd shuttle --> float seconds
for col in ["md_yard_40_dash", "md_yard_20_split", "md_yard_60_shuttle", "md_yard_10_split", "md_cone_drill", "md_yard_20_shuttle"]:
    nfl_combine_df[col + "_sec"] = nfl_combine_df[col].apply(extract_number)

# vertical/broad jump --> decimal inches
for col in ["md_vert_jump", "md_broad_jump"]:
    nfl_combine_df[col + "_in"] = nfl_combine_df[col].apply(extract_number)

# Bench press
nfl_combine_df["md_bench_press_reps"] = nfl_combine_df["md_bench_press"].apply(extract_number)

display(nfl_combine_df.head())

,md_player_name,md_player_id,md_url,md_draft_class,md_position,md_college,md_height,md_weight,md_arm_length,md_hand_size,...,md_wing_span_in,md_yard_40_dash_sec,md_yard_20_split_sec,md_yard_60_shuttle_sec,md_yard_10_split_sec,md_cone_drill_sec,md_yard_20_shuttle_sec,md_vert_jump_in,md_broad_jump_in,md_bench_press_reps
0,A'Shawn Robinson,ashawnrobinson2016,https://www.mockdraftable.com//player/ashawn-r...,2016,Defensive Tackle,Alabama,"6' 4""",307 lbs,"34.5""","10.5""",...,NaN,5.20,NaN,NaN,1.78,7.8,4.74,26.0,106.0,22.0
1,A.C. Leonard,acleonard2014,https://www.mockdraftable.com//player/ac-leona...,2014,Tight End,Tennessee State,"6' 2""",252 lbs,"33""","9.25""",...,NaN,4.50,NaN,NaN,NaN,NaN,NaN,34.0,128.0,20.0
2,A.J. Brown,ajbrown2019,https://www.mockdraftable.com//player/aj-brown...,2019,Wide Receiver,Ole Miss,"6' 0.5""",226 lbs,"32.875""","9.75""",...,78.0,4.49,NaN,NaN,NaN,NaN,NaN,36.5,120.0,19.0
3,A.J. Cann,ajcann2015,https://www.mockdraftable.com//player/aj-cann?...,2015,Offensive Guard,South Carolina,"6' 3""",313 lbs,"32.625""","10.25""",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.0
4,A.J. Derby,ajderby2015,https://www.mockdraftable.com//player/aj-derby...,2015,Tight End,Arkansas,"6' 4""",255 lbs,"30.5""","9.5""",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0


In [17]:
# Drop the original raw columns

raw_cols = [
    "md_weight", "md_height", "md_arm_length", "md_hand_size", "md_wing_span",
    "md_yard_40_dash", "md_yard_20_split", "md_yard_60_shuttle", "md_yard_10_split", "md_cone_drill", "md_yard_20_shuttle",
    "md_vert_jump", "md_broad_jump","md_bench_press"
]
nfl_combine_df.drop(columns=raw_cols, inplace=True, errors='ignore')


In [18]:
print(nfl_combine_df.dtypes)
print(nfl_combine_df.head())

md_player_name             object
md_player_id               object
md_url                     object
md_draft_class              int64
md_position                object
md_college                 object
md_weight_lbs             float64
md_height_in              float64
md_arm_length_in          float64
md_hand_size_in           float64
md_wing_span_in           float64
md_yard_40_dash_sec       float64
md_yard_20_split_sec      float64
md_yard_60_shuttle_sec    float64
md_yard_10_split_sec      float64
md_cone_drill_sec         float64
md_yard_20_shuttle_sec    float64
md_vert_jump_in           float64
md_broad_jump_in          float64
md_bench_press_reps       float64
dtype: object
     md_player_name        md_player_id  \
0  A'Shawn Robinson  ashawnrobinson2016   
1      A.C. Leonard       acleonard2014   
2        A.J. Brown         ajbrown2019   
3         A.J. Cann          ajcann2015   
4        A.J. Derby         ajderby2015   

                                              m

In [19]:
display(nfl_combine_df.head())

,md_player_name,md_player_id,md_url,md_draft_class,md_position,md_college,md_weight_lbs,md_height_in,md_arm_length_in,md_hand_size_in,md_wing_span_in,md_yard_40_dash_sec,md_yard_20_split_sec,md_yard_60_shuttle_sec,md_yard_10_split_sec,md_cone_drill_sec,md_yard_20_shuttle_sec,md_vert_jump_in,md_broad_jump_in,md_bench_press_reps
0,A'Shawn Robinson,ashawnrobinson2016,https://www.mockdraftable.com//player/ashawn-r...,2016,Defensive Tackle,Alabama,307.0,72.0,34.500,10.50,NaN,5.20,NaN,NaN,1.78,7.8,4.74,26.0,106.0,22.0
1,A.C. Leonard,acleonard2014,https://www.mockdraftable.com//player/ac-leona...,2014,Tight End,Tennessee State,252.0,72.0,33.000,9.25,NaN,4.50,NaN,NaN,NaN,NaN,NaN,34.0,128.0,20.0
2,A.J. Brown,ajbrown2019,https://www.mockdraftable.com//player/aj-brown...,2019,Wide Receiver,Ole Miss,226.0,72.0,32.875,9.75,78.0,4.49,NaN,NaN,NaN,NaN,NaN,36.5,120.0,19.0
3,A.J. Cann,ajcann2015,https://www.mockdraftable.com//player/aj-cann?...,2015,Offensive Guard,South Carolina,313.0,72.0,32.625,10.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.0
4,A.J. Derby,ajderby2015,https://www.mockdraftable.com//player/aj-derby...,2015,Tight End,Arkansas,255.0,72.0,30.500,9.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0


In [20]:
nfl_combine_df.to_csv(data_folder / 'nfl_combine_data.csv', index=False)